In [ ]:

# Step 49.2.1 - Load existing synthetic automotive data

from pyspark.sql import functions as F

# Read the existing Bronze Delta table
df_source = spark.table(
    "bronze.synthetic_vehicle_diagnostics"
)

# Count available records
total_records = df_source.count()

print(f"Source records: {total_records}")

# Display sample records
display(df_source.limit(10))

# Inspect the source schema
df_source.printSchema()



In [ ]:

# Step 49.2.2 - Initialize incremental Delta target

target_table = "silver.synthetic_vehicle_diagnostics_incremental"

# Validate source before initial load
source_count = df_source.count()
unique_ids = df_source.select("TestID").distinct().count()
null_ids = df_source.filter(F.col("TestID").isNull()).count()

print(f"Source records: {source_count}")
print(f"Unique TestIDs: {unique_ids}")
print(f"Null TestIDs: {null_ids}")

if source_count != 1000:
    raise ValueError("Unexpected source record count")

if unique_ids != source_count or null_ids > 0:
    raise ValueError("TestID validation failed")

# Create target only if it does not already exist
if not spark.catalog.tableExists(target_table):
    df_source.write \
        .format("delta") \
        .mode("errorifexists") \
        .saveAsTable(target_table)

    print("Initial Delta load completed")
else:
    print("Target already exists - initial load skipped")

# Validate target
df_target = spark.table(target_table)

print(f"Target records: {df_target.count()}")
display(df_target.limit(10))


In [ ]:
# Step 49.3.1 - Prepare incremental test batch

from datetime import datetime
from pyspark.sql import Row

# Select two existing records for controlled updates
existing = {
    row["TestID"]: row.asDict()
    for row in spark.table(
        "silver.synthetic_vehicle_diagnostics_incremental"
    ).filter(F.col("TestID").isin(1, 2)).collect()
}

if set(existing) != {1, 2}:
    raise ValueError("Expected TestIDs 1 and 2 in target")

# Preserve valid ECU, DTC and category combinations.
# Change only the test result and timestamp.
updates = []

for test_id in (1, 2):
    record = existing[test_id].copy()
    record["TestResult"] = (
        "FAIL" if record["TestResult"] == "PASS" else "PASS"
    )
    record["TestTimestamp"] = datetime(2026, 11, 7, 10, 0, 0)
    updates.append(record)

new_records = [
    {
        "TestID": 1001,
        "ECU": "ECU_Battery",
        "DTC": "P0562",
        "ErrorCategory": "Diagnostic_Error",
        "TestResult": "PASS",
        "TestTimestamp": datetime(2026, 11, 7, 10, 1, 0)
    },
    {
        "TestID": 1002,
        "ECU": "ECU_Brake",
        "DTC": "U0100",
        "ErrorCategory": "Communication_Loss",
        "TestResult": "FAIL",
        "TestTimestamp": datetime(2026, 11, 7, 10, 2, 0)
    },
    {
        "TestID": 1003,
        "ECU": "ECU_Thermal",
        "DTC": "P0118",
        "ErrorCategory": "Overtemperature",
        "TestResult": "PASS",
        "TestTimestamp": datetime(2026, 11, 7, 10, 3, 0)
    }
]

# Reuse the target schema to ensure matching data type
target_schema = spark.table(
    "silver.synthetic_vehicle_diagnostics_incremental"
).schema

df_incremental = spark.createDataFrame(
    updates + new_records,
    schema=target_schema
)

print(f"Incremental batch records: {df_incremental.count()}")
print(
    f"Distinct TestIDs: "
    f"{df_incremental.select('TestID').distinct().count()}"
)

display(df_incremental.orderBy("TestID"))


In [ ]:

# Step 49.3.2 - Incremental Data Quality Gate

from pyspark.sql import functions as F

required_columns = [
    "TestID",
    "ECU",
    "DTC",
    "ErrorCategory",
    "TestResult",
    "TestTimestamp"
]

# 1. Missing mandatory values
missing_rows = df_incremental.filter(
    " OR ".join(
        f"{column} IS NULL"
        for column in required_columns
    )
).count()

# 2. Duplicate TestIDs
duplicate_ids = (
    df_incremental
    .groupBy("TestID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 3. Invalid test results
invalid_results = df_incremental.filter(
    ~F.col("TestResult").isin("PASS", "FAIL")
).count()

# 4. Invalid DTC formats
invalid_dtcs = df_incremental.filter(
    ~F.col("DTC").rlike(r"^[PCBU][0-9A-F]{4}$")
).count()

# 5. Unexpected batch size
batch_count = df_incremental.count()
invalid_batch_size = int(batch_count != 5)

quality_issues = (
    missing_rows
    + duplicate_ids
    + invalid_results
    + invalid_dtcs
    + invalid_batch_size
)

quality_status = "PASS" if quality_issues == 0 else "FAIL"

print("=== INCREMENTAL DATA QUALITY GATE ===")
print(f"Total Batch Records: {batch_count}")
print(f"Missing Mandatory Fields: {missing_rows}")
print(f"Duplicate TestIDs: {duplicate_ids}")
print(f"Invalid Test Results: {invalid_results}")
print(f"Invalid DTC Formats: {invalid_dtcs}")
print(f"Invalid Batch Size: {invalid_batch_size}")
print(f"Quality Status: {quality_status}")

if quality_status == "FAIL":
    raise ValueError(
        "Incremental Data Quality Gate FAILED"
    )


In [ ]:
# Step 49.4.1 - Analyze incremental MERGE candidates

from pyspark.sql import functions as F

target_table = "silver.synthetic_vehicle_diagnostics_incremental"

df_target = spark.table(target_table)

# Verify the current target state
target_count = df_target.count()

# Detect duplicate keys in the target
duplicate_target_ids = (
    df_target
    .groupBy("TestID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_target_ids > 0:
    raise ValueError("Target contains duplicate TestIDs")

if df_incremental.filter(F.col("TestID").isNull()).count() > 0:
    raise ValueError("Incremental batch contains null TestIDs")

# Identify new records
df_new = df_incremental.join(
    df_target.select("TestID"),
    on="TestID",
    how="left_anti"
)

# Identify existing records
df_existing = df_incremental.join(
    df_target.select("TestID"),
    on="TestID",
    how="left_semi"
)

new_count = df_new.count()
existing_count = df_existing.count()

print("=== INCREMENTAL MERGE ANALYSIS ===")
print(f"Current Target Records: {target_count}")
print(f"New Records: {new_count}")
print(f"Existing Records: {existing_count}")
print(f"Duplicate Target IDs: {duplicate_target_ids}")

display(df_new.orderBy("TestID"))


In [ ]:
# Step 49.4.2 - Incremental Delta Lake MERGE (Upsert)

from delta.tables import DeltaTable
from pyspark.sql import functions as F

target_table = "silver.synthetic_vehicle_diagnostics_incremental"

# Safety checks
if quality_status != "PASS":
    raise ValueError("Data Quality Gate has not passed")

if df_incremental.count() != 5:
    raise ValueError("Unexpected incremental batch size")

# Load existing Delta target
delta_target = DeltaTable.forName(spark, target_table)

# Compare all business fields, using null-safe equality
business_columns = [
    "ECU",
    "DTC",
    "ErrorCategory",
    "TestResult",
    "TestTimestamp"
]

change_condition = " OR ".join(
    f"NOT (target.`{column}` <=> source.`{column}`)"
    for column in business_columns
)

# MERGE: update changed records, insert new records
(
    delta_target.alias("target")
    .merge(
        df_incremental.alias("source"),
        "target.TestID = source.TestID"
    )
    .whenMatchedUpdateAll(condition=change_condition)
    .whenNotMatchedInsertAll()
    .execute()
)

# Validate final target state
df_after_merge = spark.table(target_table)

final_count = df_after_merge.count()

duplicate_ids = (
    df_after_merge.groupBy("TestID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("=== DELTA LAKE MERGE RESULT ===")
print(f"Target Records After MERGE: {final_count}")
print(f"Duplicate TestIDs: {duplicate_ids}")

display(
    df_after_merge
    .filter(F.col("TestID").isin(1, 2, 1001, 1002, 1003))
    .orderBy("TestID")
)

if final_count != 1003:
    raise ValueError(
        f"Unexpected target count: {final_count}"
    )

if duplicate_ids > 0:
    raise ValueError("Duplicate TestIDs detected after MERGE")

print("MERGE VALIDATION: PASS")


In [ ]:
# Step 49.4.3 - Verify incremental processing idempotency

from pyspark.sql import functions as F
from delta.tables import DeltaTable

target_table = "silver.synthetic_vehicle_diagnostics_incremental"

business_columns = [
    "ECU", "DTC", "ErrorCategory",
    "TestResult", "TestTimestamp"
]

# Read the target after the first MERGE
df_before = spark.table(target_table).cache()
before_count = df_before.count()

# Identify changes before repeating the MERGE
comparison = df_incremental.alias("s").join(
    df_before.alias("t"),
    F.col("s.TestID") == F.col("t.TestID"),
    "left"
)

new_records = comparison.filter(
    F.col("t.TestID").isNull()
).count()

different_fields = [
    ~F.col(f"s.{column}").eqNullSafe(F.col(f"t.{column}"))
    for column in business_columns
]

changed_records = comparison.filter(
    F.col("t.TestID").isNotNull()
).filter(
    F.greatest(*[
        F.when(condition, 1).otherwise(0)
        for condition in different_fields
    ]) == 1
).count()

print("=== IDEMPOTENCY CHECK ===")
print(f"Records Before: {before_count}")
print(f"New Records: {new_records}")
print(f"Changed Records: {changed_records}")

# Repeat exactly the same MERGE
delta_target = DeltaTable.forName(spark, target_table)

change_condition = " OR ".join(
    f"NOT (target.`{column}` <=> source.`{column}`)"
    for column in business_columns
)

(
    delta_target.alias("target")
    .merge(
        df_incremental.alias("source"),
        "target.TestID = source.TestID"
    )
    .whenMatchedUpdateAll(condition=change_condition)
    .whenNotMatchedInsertAll()
    .execute()
)

# Read the target again after the second MERGE
df_after = spark.table(target_table)
after_count = df_after.count()

print(f"Records After: {after_count}")

assert before_count == after_count == 1003
assert new_records == 0
assert changed_records == 0

print("IDEMPOTENCY TEST: PASS")

df_before.unpersist()


In [ ]:
# Step 49.5.1 - Inspect Delta Lake MERGE history

from delta.tables import DeltaTable
from pyspark.sql import functions as F

target_table = "silver.synthetic_vehicle_diagnostics_incremental"

# Read the last 10 Delta table transactions
delta_table = DeltaTable.forName(spark, target_table)

df_history = delta_table.history(10)

# Display relevant transaction information
display(
    df_history.select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    ).orderBy(F.col("version").desc())
)


In [ ]:
# Step 49.5.2 - Analyze Delta Lake MERGE metrics

from delta.tables import DeltaTable
from pyspark.sql import functions as F

target_table = "silver.synthetic_vehicle_diagnostics_incremental"

delta_table = DeltaTable.forName(spark, target_table)

# Read MERGE transactions from Delta history
df_merge_history = (
    delta_table.history()
    .filter(F.col("operation") == "MERGE")
    .orderBy(F.col("version").asc())
)

# Extract available operation metrics
df_merge_metrics = df_merge_history.select(
    "version",
    "timestamp",
    F.coalesce(
        F.col("operationMetrics").getItem("numSourceRows").cast("long"),
        F.lit(0)
    ).alias("SourceRows"),
    F.coalesce(
        F.col("operationMetrics").getItem("numTargetRowsInserted").cast("long"),
        F.lit(0)
    ).alias("InsertedRows"),
    F.coalesce(
        F.col("operationMetrics").getItem("numTargetRowsUpdated").cast("long"),
        F.lit(0)
    ).alias("UpdatedRows"),
    F.coalesce(
        F.col("operationMetrics").getItem("numTargetRowsDeleted").cast("long"),
        F.lit(0)
    ).alias("DeletedRows")
)

print("=== DELTA MERGE AUDIT METRICS ===")
display(df_merge_metrics)

print(f"MERGE transactions found: {df_merge_metrics.count()}")


In [ ]:
# Step 49.5.3 - Create incremental processing audit table

from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType, LongType
)

audit_table = "dbo.incremental_processing_audit"

audit_schema = StructType([
    StructField("AuditID", StringType(), False),
    StructField("RunTimestamp", TimestampType(), False),
    StructField("TargetTable", StringType(), False),
    StructField("DeltaVersion", LongType(), True),
    StructField("SourceRows", LongType(), True),
    StructField("InsertedRows", LongType(), True),
    StructField("UpdatedRows", LongType(), True),
    StructField("DeletedRows", LongType(), True),
    StructField("ProcessingStatus", StringType(), False)
])

# Create the table only if it does not already exist
if not spark.catalog.tableExists(audit_table):

    empty_audit_df = spark.createDataFrame(
        [],
        audit_schema
    )

    (
        empty_audit_df.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(audit_table)
    )

    print("Audit table created successfully")

else:
    print("Audit table already exists")

# Inspect the audit table
df_audit = spark.table(audit_table)

print(f"Current audit records: {df_audit.count()}")
df_audit.printSchema()


In [ ]:
# Step 49.5.4 - Write MERGE metrics to audit table

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from datetime import datetime, timezone
from uuid import uuid4

target_table = "silver.synthetic_vehicle_diagnostics_incremental"
audit_table = "dbo.incremental_processing_audit"

# Read actual MERGE transactions
merge_history = (
    DeltaTable.forName(spark, target_table)
    .history()
    .filter(F.col("operation") == "MERGE")
    .orderBy("version")
    .collect()
)

if not merge_history:
    raise ValueError("No MERGE transactions found")

# Read already logged transaction versions
logged_versions = {
    row["DeltaVersion"]
    for row in spark.table(audit_table)
        .filter(F.col("TargetTable") == target_table)
        .select("DeltaVersion")
        .collect()
}

required_metrics = [
    "numSourceRows",
    "numTargetRowsInserted",
    "numTargetRowsUpdated",
    "numTargetRowsDeleted"
]

audit_records = []

for operation in merge_history:
    version = int(operation["version"])

    # Skip transactions already recorded
    if version in logged_versions:
        continue

    metrics = operation["operationMetrics"] or {}

    # Do not interpret missing metrics as zero
    missing_metrics = [
        key for key in required_metrics
        if key not in metrics or metrics[key] is None
    ]

    if missing_metrics:
        print(
            f"Skipping version {version}: "
            f"missing metrics {missing_metrics}"
        )
        continue

    audit_records.append({
        "AuditID": str(uuid4()),
        "RunTimestamp": datetime.now(timezone.utc).replace(tzinfo=None),
        "TargetTable": target_table,
        "DeltaVersion": version,
        "SourceRows": int(metrics["numSourceRows"]),
        "InsertedRows": int(metrics["numTargetRowsInserted"]),
        "UpdatedRows": int(metrics["numTargetRowsUpdated"]),
        "DeletedRows": int(metrics["numTargetRowsDeleted"]),
        "ProcessingStatus": "SUCCESS"
    })

# Append only previously unlogged transactions
if audit_records:
    df_new_audit = spark.createDataFrame(
        audit_records,
        schema=audit_schema
    )

    (
        df_new_audit.write
        .format("delta")
        .mode("append")
        .saveAsTable(audit_table)
    )

    print(f"Audit records inserted: {len(audit_records)}")
else:
    print("No new MERGE transactions to audit")

# Display audit results
display(
    spark.table(audit_table)
    .filter(F.col("TargetTable") == target_table)
    .orderBy(F.col("DeltaVersion").asc())
)


In [ ]:
# Step 49.5.5 - Validate audit records against Delta history

from delta.tables import DeltaTable
from pyspark.sql import functions as F

target_table = "silver.synthetic_vehicle_diagnostics_incremental"
audit_table = "dbo.incremental_processing_audit"

# Read MERGE transaction history
df_history = (
    DeltaTable.forName(spark, target_table)
    .history()
    .filter(F.col("operation") == "MERGE")
    .select(
        F.col("version").alias("DeltaVersion"),
        F.col("operationMetrics").alias("Metrics")
    )
)

# Read audit records
df_audit = (
    spark.table(audit_table)
    .filter(F.col("TargetTable") == target_table)
)

# Detect duplicate audit entries per Delta version
df_duplicates = (
    df_audit.groupBy("DeltaVersion")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = df_duplicates.count()

# Find MERGE versions missing from the audit table
df_missing = df_history.join(
    df_audit.select("DeltaVersion"),
    on="DeltaVersion",
    how="left_anti"
)

missing_count = df_missing.count()

print("=== AUDIT VALIDATION ===")
print(f"MERGE Transactions: {df_history.count()}")
print(f"Audit Records: {df_audit.count()}")
print(f"Duplicate Audit Versions: {duplicate_count}")
print(f"Unaudited MERGE Versions: {missing_count}")

display(df_missing.orderBy("DeltaVersion"))

if duplicate_count > 0:
    raise ValueError("Duplicate audit versions detected")

if missing_count > 0:
    print(
        "WARNING: Some MERGE transactions are not audited. "
        "Check Delta history and available operation metrics."
    )
else:
    print("AUDIT VALIDATION: PASS")


In [ ]:
# Step 49.5.6 - Verify audit metrics against Delta history

from delta.tables import DeltaTable
from pyspark.sql import functions as F

target_table = "silver.synthetic_vehicle_diagnostics_incremental"
audit_table = "dbo.incremental_processing_audit"

metric_mapping = {
    "SourceRows": "numSourceRows",
    "InsertedRows": "numTargetRowsInserted",
    "UpdatedRows": "numTargetRowsUpdated",
    "DeletedRows": "numTargetRowsDeleted"
}

history = (
    DeltaTable.forName(spark, target_table)
    .history()
    .filter(F.col("operation") == "MERGE")
    .select(
        F.col("version").alias("DeltaVersion"),
        "operationMetrics"
    )
)

audit = (
    spark.table(audit_table)
    .filter(F.col("TargetTable") == target_table)
)

joined = history.alias("h").join(
    audit.alias("a"),
    on="DeltaVersion",
    how="inner"
)

# Compare each persisted metric with the original metric
conditions = [
    F.col(f"a.{audit_col}") !=
    F.col("h.operationMetrics").getItem(delta_key).cast("long")
    for audit_col, delta_key in metric_mapping.items()
]

missing_metrics = [
    F.col("h.operationMetrics").getItem(delta_key).isNull()
    for delta_key in metric_mapping.values()
]

mismatch_condition = conditions[0]

for condition in conditions[1:] + missing_metrics:
    mismatch_condition = mismatch_condition | condition

mismatch_count = joined.filter(mismatch_condition).count()

print("=== AUDIT METRIC VALIDATION ===")
print(f"Compared Transactions: {joined.count()}")
print(f"Metric Mismatches: {mismatch_count}")

if mismatch_count > 0:
    raise ValueError("Audit metrics differ from Delta history")

print("AUDIT METRICS VALIDATION: PASS")
